# Dynamic Workflows — Claude Managed Agents

In module 06 the lead agent delegated to subagents itself: it sent each task, waited for each report, and merged the results in its own context. That works for a few specialists. It doesn't scale to one agent per document across hundreds of documents, or to a reviewer that checks every result.

With **dynamic workflows**, the agent **writes a program** — a *workflow* — that runs many agents and combines what they return. The server executes it in the background as a **workflow run**:

```
workflow run  (the program the agent wrote; the server runs it in the background)
 ├─ phase "Read"     →  agent thread × N   (fan out: one agent per contract, in parallel)
 └─ phase "Verify"   →  agent thread × N   (each checks a reader's result)
                                ↓
        results flow from agent to agent through the program, not through the lead's context
```

What changes for you:

| | Subagents (06) | Dynamic workflows (07) |
|---|---|---|
| Who coordinates | The lead, turn by turn | A program the lead writes |
| Where results go | Into the lead's context | Into the program, which passes them on |
| Follow-ups | The lead can message a subagent again | No; the server archives each thread by the end of the run |
| You follow | Thread events | `workflow_run.*` events: the run, its phases, its threads |
| Starting it | The lead decides | The lead decides; **no API call starts a run** |

Everything from the earlier modules comes together here: files (04) as the material, answering on arrival (03), budgets and `stop_reason` (05), and threads (06).

## 1. Setup

In [1]:
%pip install -q -U "anthropic>=1.13" python-dotenv


[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: /usr/local/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


In [2]:
import time
from pathlib import Path

import anthropic
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())

client = anthropic.Anthropic()
environment = client.beta.environments.create(
    name="dynamic-workflows-env",
    config={"type": "cloud", "networking": {"type": "unrestricted"}},
)
print(f"Environment: {environment.id}")

Environment: env_01FaCBEJ2fSNXww4W223YeuX


## 2. A Predefined Agent for the Workflow

A workflow runs two kinds of agent:

- **Inline agents**, which the workflow defines itself. It writes a system prompt for each one, and each uses the lead's model and tools.
- **Predefined agents**, which you created and listed in **`workflows.predefined_agents`** (up to 20). Each keeps its own model, prompt, and tools.

Reading one contract and pulling out its terms is a narrow, repetitive job, so we give the workflow a **Contract Reader** on `claude-haiku-5-5`. The checking step is left to inline agents on the lead's model. Every agent in a run costs tokens, so this choice matters at scale.

In [3]:
reader = client.beta.agents.create(
    name="Contract Reader",
    description="Reads one supplier contract and extracts its key terms.",
    model="claude-haiku-5-5",
    system=(
        "You read exactly one supplier contract and extract its key terms: supplier, effective date, "
        "auto-renewal, non-renewal notice period, termination for convenience, liability cap and carve-outs, "
        "change-of-control clause, governing law. Quote the clause number for every term. "
        "Say 'not stated' when the contract is silent. Reply with the terms only."
    ),
    tools=[{
        "type": "agent_toolset_20260401",
        "configs": [{"name": "web_search", "enabled": False}, {"name": "web_fetch", "enabled": False}],
    }],
)
print(f"Reader: {reader.id}  v{reader.version}  {reader.model.id}")

Reader: agent_01VKXB63zctrnP2in3x7pTaA  v1  claude-haiku-5-5


## 3. Turn On Dynamic Workflows

Set `multiagent.type` to `multiagent_20261001` and enable **`workflows`**. Workflows are on by default with this type; we write it out to be clear, and to attach the reader. We switch **`subagents` off**, so the lead hands out work only through runs.

The **system prompt** matters most here. The lead decides *whether* and *how* to run a workflow, so tell it:

- **when** to use a run (more than a few contracts) and when not to (one or two);
- **how to split the work**: one reader per contract, then a check of *every* extraction, with failures redone;
- **what to do when an agent fails**: list the contract as not covered instead of failing the run;
- **a time limit** for the run (the default lifetime is 24 hours).

> Starting a run is not governed by permission policies. Those still apply to the tools that a run's agents call.

In [4]:
LEAD_SYSTEM = "You review supplier contracts for Brightloom Inc. Contracts are mounted read-only under /mnt/session/uploads/contracts/. When you're asked to review more than a few contracts, start a workflow run instead of reading them yourself. In the run: first, one Contract Reader per contract extracts its key terms. Then a second agent checks every extraction against its contract, not only the ones that look wrong, and looks for anything the reader missed; an extraction that fails the check is redone once. One agent failing must not fail the run: list that contract as not covered. Give the run a 30-minute lifetime. When the run ends, write /mnt/session/outputs/summary.csv (one row per contract) and /mnt/session/outputs/report.md (risks ranked from most to least urgent, citing clause numbers). Review one or two contracts yourself, without a run. Keep chat replies short."

lead = client.beta.agents.create(
    name="Contract Review Lead (workflows)",
    model="claude-opus-5-5",
    system=LEAD_SYSTEM,
    tools=[{"type": "agent_toolset_20260401"}],
    multiagent={
        "type": "multiagent_20261001",
        "workflows": {
            "type": "enabled",
            "predefined_agents": [{"type": "agent", "id": reader.id, "version": reader.version}],
        },
        "subagents": {"type": "disabled"},
    },
)
print(f"Lead: {lead.id}  v{lead.version}")
print(lead.multiagent)

Lead: agent_01NFQGpfxKxyMU1NCgumAKoj  v1
BetaManagedAgentsMultiagent20261001(advisor=BetaManagedAgentsMultiagentAdvisorDisabled(type='disabled'), subagents=BetaManagedAgentsMultiagentSubagentsDisabled(type='disabled'), type='multiagent_20261001', workflows=BetaManagedAgentsMultiagentWorkflowsEnabled(inline_agents=BetaManagedAgentsMultiagentInlineAgentsEnabled(type='enabled'), predefined_agents=[BetaManagedAgentsAgentReference(id='agent_01VKXB63zctrnP2in3x7pTaA', type='agent', version=1)], type='enabled'))


## 4. Mount the Contracts and Start a Budgeted Session

Every agent in a run uses tokens, so **set a budget when you create the session**: it caps the whole session, runs included. A run that reaches the cap **pauses**, with its threads going idle at `budget_reached`, and resumes when you raise the cap (module 05).

All of a run's threads work in the **same sandbox**, so they read the mounted contracts directly.

In [5]:
uploaded = {}
for path in sorted(Path("../sample-data/contracts").glob("*.md")):
    uploaded[path.name] = client.files.upload(file=path).id

session = client.beta.sessions.create(
    agent={"type": "agent", "id": lead.id, "version": lead.version},
    environment_id=environment.id,
    title="Contract review with a workflow run",
    resources=[
        {"type": "file", "file_id": file_id, "mount_path": f"/contracts/{name}"}
        for name, file_id in uploaded.items()
    ],
    budget={"type": "limit", "max_list_cost": {"amount": "500", "currency": "USD"}},
)
print(f"Session: {session.id}  ({len(uploaded)} contracts, budget $5.00)")

Session: sesn_016NBzWSgxEN9q4o92RBv8ao  (6 contracts, budget $5.00)


## 5. Follow the Run

Run events arrive on the session's stream (the primary thread):

| Event | Meaning |
|---|---|
| `workflow_run.created` | The lead started a run. It carries `name`, `description`, and the declared `phases` (each with `id` and `name`). |
| `workflow_run.status_running` / `status_idle` | The run is executing / paused (for example at the budget) |
| `workflow_run.phase_started` / `phase_ended` | The run entered or left a phase. These carry only `workflow_run_phase_id`, so look the name up in `phases`. |
| `session.thread_created` with `workflow_run_id` | The run started an agent thread |
| `session.thread_status_terminated` | A thread finished; the server archives a run's threads itself |
| `workflow_run.error` | A run error, or a start the server refused (then `workflow_run_id` is `null`). Log it, but don't treat it as the end. |
| `workflow_run.status_ended` | The run ended, with a `result`: `completed`, `stopped`, or `error` (`timeout_error`, `program_error`, …) |

**When is the work done?** Not at the first idle. While a run is open, the lead can end its turn and the session can go idle. When the run ends, the lead gets a new turn to read the result and write the deliverables. The work is done when **both** of these are true:

1. every run you've seen created has its `workflow_run.status_ended`, and
2. after that, `session.status_idle` arrives with `stop_reason: end_turn`.

Two more habits carry over from earlier modules. A run thread's tool call can wait on your client while the session stays `running`, so **answer custom tools and confirmations when their events arrive** (module 03). This agent has neither, so the loop below only notes where they would go. And stop on `budget_reached`, `retries_exhausted`, or `refusal` so you can handle them.

In [6]:
def text_of(event) -> str:
    return "".join(block.text for block in event.content if block.type == "text")


def follow_runs(stream) -> str:
    """Follow the session until every run has ended and the lead has finished its turn."""
    open_runs = {}     # workflow_run_id -> run name
    phase_names = {}   # (run id, phase id) -> phase name
    threads = {}       # workflow_run_id -> [created, terminated]
    run_of = {}        # session_thread_id -> workflow_run_id
    t0 = time.monotonic()

    def log(line: str) -> None:
        print(f"{time.monotonic() - t0:6.0f}s  {line}")

    for event in stream:
        match event.type:
            # --- the run
            case "workflow_run.created":
                open_runs[event.workflow_run_id] = event.name
                threads[event.workflow_run_id] = [0, 0]
                for phase in event.phases:
                    phase_names[event.workflow_run_id, phase.id] = phase.name
                log(f"RUN CREATED  {event.name!r}  phases: {[p.name for p in event.phases]}")
            case "workflow_run.status_running":
                log("run running")
            case "workflow_run.status_idle":
                log("run paused (idle)")
            case "workflow_run.phase_started":
                name = phase_names.get((event.workflow_run_id, event.workflow_run_phase_id), event.workflow_run_phase_id)
                log(f"PHASE STARTED  {name}")
            case "workflow_run.phase_ended":
                name = phase_names.get((event.workflow_run_id, event.workflow_run_phase_id), event.workflow_run_phase_id)
                created, done = threads.get(event.workflow_run_id, [0, 0])
                log(f"phase ended    {name}   (threads so far: {done}/{created} finished)")
            case "workflow_run.error":
                log(f"run error: {event.error.type}: {event.error.message}")
            case "workflow_run.status_ended":
                name = open_runs.pop(event.workflow_run_id, event.workflow_run_id)
                result = event.result.type
                if result == "error":
                    result += f" ({event.result.error.type})"
                created, done = threads.get(event.workflow_run_id, [0, 0])
                log(f"RUN ENDED    {name!r}: {result}   ({created} agent threads)")

            # --- the run's agent threads
            case "session.thread_created" if event.workflow_run_id:
                threads.setdefault(event.workflow_run_id, [0, 0])[0] += 1
                run_of[event.session_thread_id] = event.workflow_run_id
                log(f"  + thread {event.agent_name}")
            case "session.thread_status_terminated" if event.session_thread_id in run_of:
                threads[run_of[event.session_thread_id]][1] += 1

            # --- calls that wait on your client: answer on arrival (module 03)
            case "agent.custom_tool_use":
                log(f"custom tool call {event.name}: answer it here with user.custom_tool_result")

            # --- the lead itself
            case "agent.message":
                log(f"Lead: {text_of(event)}")
            case "session.status_idle":
                reason = event.stop_reason.type
                if reason == "requires_action":
                    continue
                if reason == "end_turn" and open_runs:
                    log("lead ended its turn while the run works in the background — keep following")
                    continue
                log(f"session idle: {reason}")
                return reason
            case "session.status_terminated":
                log("session terminated")
                return "terminated"

In [7]:
with client.beta.sessions.events.stream(session.id) as stream:
    client.beta.sessions.events.send(
        session.id,
        events=[{"type": "user.message", "content": [{"type": "text", "text":
            "Review all the contracts in /mnt/session/uploads/contracts/ and write the summary and the report."}]}],
    )
    stop_reason = follow_runs(stream)

    27s  RUN CREATED  'Brightloom supplier contract review'  phases: ['Extract key terms', 'Check every extraction', 'Redo failed extractions']


    28s  run running
    28s  PHASE STARTED  Extract key terms
    28s    + thread Contract Reader
    28s    + thread Contract Reader
    28s    + thread Contract Reader
    28s    + thread Contract Reader
    28s    + thread Contract Reader
    28s    + thread Contract Reader


    29s  Lead: I've started a review run for all 6 contracts, with a 30-minute limit. Once it ends I'll write `summary.csv` and `report.md`.

<br>


    54s  phase ended    Extract key terms   (threads so far: 6/6 finished)
    54s  PHASE STARTED  Check every extraction
    54s    + thread Extraction Checker
    54s    + thread Extraction Checker
    54s    + thread Extraction Checker
    54s    + thread Extraction Checker
    54s    + thread Extraction Checker
    54s    + thread Extraction Checker


    75s  phase ended    Check every extraction   (threads so far: 12/12 finished)
    75s  PHASE STARTED  Redo failed extractions
    76s    + thread Contract Reader
    76s    + thread Contract Reader


   102s  phase ended    Redo failed extractions   (threads so far: 14/14 finished)


   104s  RUN ENDED    'Brightloom supplier contract review': completed   (14 agent threads)


   167s  Lead: I've reviewed all 6 contracts and written the summary and the report. Every extraction was checked against its contract. Tessaly and Orrin failed the first check and were redone; the redos weren't re-checked by an agent, but I read both contracts and they match.

**Most urgent risks:**
1. **Halvard Freight:** the contract renews for another year unless notice is given by about **2 Nov 2026**, three weeks away (clause 1.2). The rate schedule (Schedule A) is also missing from the file.
2. **Tessaly Cleaning:** the contract is now month to month, so Tessaly can stop HQ cleaning on 30 days' notice (clause 3.1). There's also no confidentiality clause, although its staff are in HQ every night.
3. **Corvel Analytics:** Corvel can raise prices by any amount at renewal (clause 2.2). Notice of a rise could come by about 14 Jan 2027, and the cancellation deadline is about 13 Feb 2027.
4. **Mirefield Packaging:**
   - Brightloom must buy at least 250,000 units a year or pay a fee fo

Read the trace. The lead started a run with **phases it chose**. Agent threads fanned out across the contracts, mostly as `Contract Reader` from our predefined list, plus inline agents for the checks. The run ended, and then the lead got a turn to read the result and write the files. Notice what **you didn't see**: the workflow's code, the calls the lead made to start and manage the run, and what each thread returned to the program. Those stay inside the run.

## 6. Inspect the Run's Threads

A run's threads are ordinary session threads with a **`workflow_run_id`**. For a predefined agent, `agent` carries its `id` and `version`. For an inline agent, `agent.type` is `inline` and carries the name and system prompt that the workflow wrote. Each thread has its own `usage`, priced at its own model.

In [8]:
def cents(amount) -> str:
    return f"${int(amount.amount) / 100:.2f}"


run_threads = [t for t in client.beta.sessions.threads.list(session.id) if t.workflow_run_id]
for t in run_threads:
    kind = t.agent.type
    name = getattr(t.agent, "name", None) or "-"
    cost = cents(t.usage.list_cost) if t.usage and t.usage.list_cost else "-"
    print(f"{t.id}  {kind:<8} {name:<28} {t.status:<11} {cost}")

s = client.beta.sessions.retrieve(session.id)
print(f"\n{len(run_threads)} run threads — session total {cents(s.usage.list_cost)} of {cents(s.budget.max_list_cost)} budget")

sthr_01XXvVkD2F8fYU3Hf772WpPF  agent    Contract Reader              terminated  $0.01
sthr_019e1h6PSs1wjSX7Gd73588u  agent    Contract Reader              terminated  $0.02
sthr_01Gk9aVTTMV8RZJvLovPSVjc  agent    Contract Reader              terminated  $0.01
sthr_01MQiqsToYbPyNXKUfBgQCU8  agent    Contract Reader              terminated  $0.01
sthr_019aRa8magsTvxcWdEHt7kzD  agent    Contract Reader              terminated  $0.01
sthr_01TcGLBk2Kk7cBZEZyxKmoYn  agent    Contract Reader              terminated  $0.01
sthr_016ZkpzQcG36xGJeduMGQjSQ  inline   Extraction Checker           terminated  $0.09
sthr_01RyNrUYz5fUzMFV3bktVQgG  inline   Extraction Checker           terminated  $0.07
sthr_011UNzSWd9P4LVjuUAXxnmtF  inline   Extraction Checker           terminated  $0.08
sthr_01TJQ9KYxPhqrERDAXzRdGTz  inline   Extraction Checker           terminated  $0.07
sthr_01BhMfWTQ9vKYNDXB2tBXruu  inline   Extraction Checker           terminated  $0.08
sthr_01T9zaopCoWtqCCetdvEE9Ju  inline   Ext

Drill into one thread. Its first `agent.thread_message_received` is **the prompt the workflow wrote** for it. The API doesn't guarantee that text, so read it but don't parse it.

In [9]:
thread = next((t for t in run_threads if getattr(t.agent, "name", "") == "Contract Reader"), run_threads[0])
for event in client.beta.sessions.threads.events.list(thread.id, session_id=session.id):
    match event.type:
        case "agent.thread_message_received":
            print(f"[prompt from the workflow]\n{text_of(event)[:600]}\n")
        case "agent.tool_use":
            print(f"[{event.name}] {str(event.input.get('file_path') or event.input.get('command') or '')[:90]}")
        case "agent.message":
            print(f"\n[reply]\n{text_of(event)[:600]}")

[prompt from the workflow]
Brightloom Inc. is reviewing its supplier contracts. The output will be a one-row-per-contract CSV summary and a report ranking risks to Brightloom from most to least urgent, each citing clause numbers. Contracts are read-only markdown files; change no files.

Please extract the key terms of the contract at /mnt/session/uploads/contracts/04-tessaly-cleaning.md. Quote dates, amounts and notice periods exactly as written, and cite the clause number for every risk. Write 'not stated' for a field the contract does not cover rather than guessing. In `risks`, list each term that exposes Brightloom (

[read] /mnt/session/uploads/contracts/04-tessaly-cleaning.md


## 7. Ask for the Workflow It Wrote

You can't see a workflow's code on the stream, but you can **ask the agent for it** once the run has ended and the session is idle. Ask it not to start another run.

In [10]:
reply = []
with client.beta.sessions.events.stream(session.id) as stream:
    client.beta.sessions.events.send(
        session.id,
        events=[{"type": "user.message", "content": [{"type": "text", "text":
            "Print the workflow that you started the run with, word for word, in one code block. "
            "Do not start another run."}]}],
    )
    for event in stream:
        if event.type == "agent.message":
            reply.append(text_of(event))
        elif event.type == "session.status_idle" and event.stop_reason.type != "requires_action":
            break

from IPython.display import Markdown, display
display(Markdown("\n".join(reply)))

```javascript
let dir = "/mnt/session/uploads/contracts/";
let files = ["01-halvard-freight.md","02-corvel-analytics.md","03-mirefield-packaging.md","04-tessaly-cleaning.md","05-quillon-cloud.md","06-orrin-staffing.md"];
let context = "Brightloom Inc. is reviewing its supplier contracts. The output will be a one-row-per-contract CSV summary and a report ranking risks to Brightloom from most to least urgent, each citing clause numbers. Contracts are read-only markdown files; change no files.";
let riskItem = { type: "object", properties: { clause: { type: "string" }, risk: { type: "string" }, severity: { type: "string", enum: ["high","medium","low"] }, urgency_note: { type: "string" } }, required: ["clause","risk","severity"] };
let extractSchema = { type: "object", properties: {
  supplier: { type: "string" }, service: { type: "string" }, contract_value: { type: "string" },
  start_date: { type: "string" }, end_date: { type: "string" }, auto_renewal: { type: "string" },
  termination_notice: { type: "string" }, payment_terms: { type: "string" }, price_changes: { type: "string" },
  liability_cap: { type: "string" }, indemnity: { type: "string" }, data_confidentiality: { type: "string" },
  governing_law: { type: "string" }, other_terms: { type: "string" },
  risks: { type: "array", items: riskItem }, blocked: { type: "string" } },
  required: ["supplier","start_date","end_date","auto_renewal","termination_notice","payment_terms","liability_cap","risks"] };
let checkSchema = { type: "object", properties: { pass: { type: "boolean" }, errors: { type: "array", items: { type: "string" } }, missed: { type: "array", items: { type: "string" } }, notes: { type: "string" } }, required: ["pass","errors","missed"] };
let checkerSystem = "In this run you check another agent's extraction of a Brightloom Inc. supplier contract before it reaches Brightloom's reviewers. Read the contract yourself, in full, and compare every field and every cited clause number to the text. A wrong date, notice period, cap or clause number costs more than a missing nicety. Also look for terms the reader missed that matter to Brightloom (auto-renewal traps, uncapped liability, unilateral price rises, data/IP terms, exclusivity, penalties). Read only; change no files.";
function readerBrief(f, fix) {
  return context + "\n\nPlease extract the key terms of the contract at " + dir + f + ". Quote dates, amounts and notice periods exactly as written, and cite the clause number for every risk. Write 'not stated' for a field the contract does not cover rather than guessing. In `risks`, list each term that exposes Brightloom (e.g. auto-renewal with a notice deadline, uncapped or one-sided liability, unilateral price increases, weak data protection), with severity and, where a date or deadline drives urgency, say which in urgency_note." + (fix ? "\n\nA previous extraction of this contract failed a check. The checker reported these problems; please verify each against the contract and correct them, and include anything listed as missed:\n" + fix : "");
}
return await step.workflow({
  name: "Brightloom supplier contract review",
  description: "Extract key terms from each of 6 supplier contracts, check every extraction against its contract, redo those that fail once.",
  phases: [
    { title: "Extract key terms", detail: "One Contract Reader per contract extracts its key terms and risks", run: async (step) => {
      let rs = await step.parallel(files, (f) => step.agent({ agent: "Contract Reader", brief: readerBrief(f, ""), schema: extractSchema }));
      return files.map((f, i) => ({ file: f, ok: rs[i].ok, extraction: rs[i].ok ? rs[i].value : null, error: rs[i].ok ? null : String(rs[i].reason).slice(0, 300) }));
    } },
    { title: "Check every extraction", detail: "A second agent checks each extraction against its contract and looks for missed terms", run: async (step, ex) => {
      let todo = ex.filter(x => x.ok);
      let rs = await step.parallel(todo, (x) => step.agent({ define: { name: "Extraction Checker", system: checkerSystem }, brief: "Contract: " + dir + x.file + "\n\nThe reader's extraction (JSON):\n" + JSON.stringify(x.extraction) + "\n\nSet pass to false if any field or clause citation is wrong or a material term is missed; list each wrong item in `errors` and each missed term (with its clause number) in `missed`.", schema: checkSchema }));
      return ex.map(x => { let j = todo.indexOf(x); if (j < 0) return Object.assign({}, x, { check: null }); let r = rs[j]; return Object.assign({}, x, { check: r.ok ? r.value : null, check_error: r.ok ? null : String(r.reason).slice(0, 300) }); });
    } },
    { title: "Redo failed extractions", detail: "Each extraction that failed its check is redone once with the checker's notes", run: async (step, checked) => {
      let redo = checked.filter(x => x.ok && x.check && !x.check.pass);
      let rs = await step.parallel(redo, (x) => step.agent({ agent: "Contract Reader", brief: readerBrief(x.file, JSON.stringify({ errors: x.check.errors, missed: x.check.missed, notes: x.check.notes || "" })), schema: extractSchema }));
      let results = checked.map(x => {
        let j = redo.indexOf(x);
        let base = { file: x.file, extraction: x.extraction, check: x.check, check_error: x.check_error || null, redone: false, redo_failed: false };
        if (j >= 0) { if (rs[j].ok) { base.extraction = rs[j].value; base.redone = true; } else { base.redo_failed = true; } }
        return base;
      });
      let covered = results.filter((r, i) => checked[i].ok);
      let not_covered = checked.filter(x => !x.ok).map(x => ({ file: x.file, reason: x.error }));
      return { covered: covered, not_covered: not_covered, unchecked: checked.filter(x => x.ok && !x.check).map(x => x.file), redone: results.filter(r => r.redone).map(r => r.file), redo_failed: results.filter(r => r.redo_failed).map(r => r.file) };
    } }
  ] });
```

This is the program the server ran: its phases, how it fans out, how it passes each reader's result to a checker, and how it handles a failed agent. The lead wrote it for *this* request. A different request gets a different workflow.

## 8. Download the Deliverables

In [11]:
EXPECTED = {"summary.csv", "report.md"}
outputs = {}
for _ in range(20):   # outputs can lag the idle by a few seconds
    outputs = {f.filename: f for f in client.beta.files.list(scope_id=session.id, betas=["managed-agents-2026-04-01"])
               if f.downloadable}
    if EXPECTED <= outputs.keys():
        break
    time.sleep(3)

out_dir = Path("outputs")
out_dir.mkdir(exist_ok=True)
for name, f in outputs.items():
    client.files.download(f.id).write_to_file(out_dir / name)
    print(f"{name:<14} {f.size_bytes:>6} bytes  ->  {out_dir / name}")

if (out_dir / "report.md").exists():
    display(Markdown((out_dir / "report.md").read_text()))

summary.csv      3865 bytes  ->  outputs/summary.csv


report.md        7243 bytes  ->  outputs/report.md


# Brightloom supplier contract review: risks by urgency

Review date: 11 October 2026. Contracts covered: 6 of 6, none missing.

How the review was done: one Contract Reader per contract pulled out the key terms. A second agent then checked every extraction against its contract. Four passed. Two failed: Tessaly had a misread termination clause and outdated dates, and Orrin had a wrong reading of clause 5.1 and the liability cap. Both were redone once. The redone versions were not checked by an agent a second time. I read both contracts myself and the redone versions match the text. I also corrected the Corvel renewal dates, which the reader had worked out for the 2026 renewal that has already passed.

Contracts rarely give an end date, so most dates below are worked out from the term length. Before acting on a deadline, confirm the exact date with the contract owner.

---

## 1. Halvard Freight: act now (deadline 2 Nov 2026)
- **Cl. 1.2: auto-renewal.** The contract renews for one year unless notice is given at least 90 days before the term ends. The initial term is assumed to end 31 Jan 2027, so **notice is due by about 2 Nov 2026, three weeks from now** (3 Nov if the term ends 1 Feb 2027). After that, the only way out is cl. 3.2: 60 days' notice plus a fee of two months' average fees.
- **Cl. 2.1: Schedule A (the rates) is missing from the file.** The contract value can't be checked. Get Schedule A before deciding on renewal.
- **Cl. 4.2: indemnity obligations sit outside the liability cap, but the contract never says what they are.** This makes exposure unclear for both sides.
- **No data-protection or confidentiality clause.** Cl. 4.2 refers to "confidentiality" but no clause defines it. Freight data may include personal data.
- Cl. 4.1: liability is capped at 12 months' fees, which may be low for cargo-loss claims. Cl. 5.1: Halvard can assign the contract on a merger or asset sale without consent, and Brightloom gets no right to leave. Cl. 2.2: one price rise a year, capped at US CPI, with no notice period stated.

## 2. Tessaly Cleaning: risk is live today
- **Cl. 3.1 with 1.1: Tessaly can leave at any time on 30 days' notice, with no penalty.** The initial term ended about 31 Aug 2026 and the contract is now month to month. Nightly HQ cleaning could stop with 30 days' warning, and Tessaly can use that threat to push for a higher fee. Line up a backup supplier.
- **No confidentiality clause, although cleaners are in HQ every night.** Cl. 5.1 requires background checks only.
- **Cl. 4.2 with 4.1:** Tessaly's liability for property damage is uncapped, but its required insurance is only USD 2M per occurrence. There is no certificate requirement and no aggregate limit.
- No indemnity for third-party claims, such as an injury caused by Tessaly staff.

## 3. Corvel Analytics: uncapped price rise at the next renewal (Jan–Feb 2027)
- **Cl. 2.2: Corvel can raise fees by any amount at renewal, with 60 days' notice.** The contract already renewed on 15 Mar 2026. For the 15 Mar 2027 renewal, notice of a price rise could arrive by about 14 Jan 2027.
- **Cl. 1.1: cancellation is due at least 30 days before renewal, so by about 13 Feb 2027.** That leaves about a month to react to a new price.
- **Cl. 4.1: Corvel's liability is capped at only 3 months' fees.** The cap protects Corvel only, and nothing is excluded from it. Cl. 4.2 adds a mutual exclusion of consequential damages.
- **Cl. 3.2: data is deleted 30 days after termination unless an export is requested first.** Request the export before giving any cancellation notice.
- Cl. 3.1: Corvel may use de-identified usage data. There are no security or breach-notification terms, no termination for convenience and no indemnity. Fees are paid a year in advance (cl. 2.1).
- In Brightloom's favour: cl. 5.1 lets Brightloom leave with a pro-rata refund if Corvel changes ownership.

## 4. Mirefield Packaging: minimum-volume commitment, and supply ends 31 May 2027
- **Cl. 2.1–2.2: minimum purchase commitment.** Brightloom must buy at least 250,000 units per contract year (1 Jun–31 May) or pay EUR 0.40 for each unit short. Check this year's volumes against the minimum.
- **Cl. 3.2: Brightloom cannot terminate for convenience,** so it is held to the volume commitment until the end of the term. The exits are breach with a 45-day cure period (cl. 3.1) or a Mirefield change of control (cl. 5.1, 90 days' notice).
- **Cl. 1.1: the contract does not renew. It expires about 31 May 2027,** and there is no renewal mechanism. Start negotiating a replacement or new contract early so packaging supply doesn't stop.
- **Cl. 4.1–4.2: only Mirefield's liability is capped** (EUR 500k), and its remedy for defects is limited to replacement or refund. Brightloom's own liability is not capped.
- Cl. 5.1: if Brightloom changes ownership, it must notify Mirefield within 10 business days, and Mirefield may then leave on 90 days' notice. The contract has no price, payment or confidentiality terms.

## 5. Quillon Cloud: costly lock-in (deadline about 4 Jul 2027)
- **Cl. 1.1: auto-renews for two years unless notice is given at least 180 days before the term ends.** The initial term ends about 31 Dec 2027, so **notice is due by about 4 Jul 2027.**
- **Cl. 3.1: early exit means paying all fees for the rest of the term.** After a renewal, that becomes two years of fees.
- **Cl. 2.1–2.2: service credits are the only remedy for downtime,** capped at 30% of the monthly fee. That is thin cover for the system that runs order management.
- Cl. 4.1: Quillon's liability is capped at 6 months' fees. Cl. 4.2 takes data-protection obligations outside the cap, but the contract never defines them. Cl. 5.1: Quillon can assign the contract to an affiliate or acquirer without consent. No payment, price-change, indemnity or termination-for-breach terms.

## 6. Orrin Staffing: deadline about 1 Aug 2027
- **Cl. 1.1: auto-renews for one year unless 60 days' notice is given.** The 18-month initial term ends about 30 Sep 2027, so notice is due by about 1 Aug 2027.
- **Cl. 3.1: either side can leave on 30 days' notice,** so Orrin could pull staff during peak season.
- **Cl. 2.2: hiring an Orrin worker directly within 6 months of their last assignment costs 20% of first-year salary.**
- **Cl. 2.1: Schedule 1 (the bill rates) is missing.** Rates, payment terms and any price-change rules can't be checked.
- Cl. 4.1–4.2: Orrin's indemnity covers wage-and-hour claims only. It most likely sits outside the USD 1M cap, which is good for Brightloom; legal should confirm this reading. No confidentiality or data clause. Cl. 5.1: if Orrin changes ownership, Brightloom may leave on 15 days' notice; no deadline is set for using that right.

---

## Patterns across contracts
- **Data and confidentiality are missing or undefined in all six contracts.** Halvard, Mirefield, Tessaly and Orrin have none. Quillon and Halvard refer to such obligations without defining them. Corvel's are partial.
- **Rate schedules are missing:** Halvard Schedule A and Orrin Schedule 1.
- **Four auto-renewal deadlines to put in the calendar:** Halvard 2 Nov 2026, Corvel 13 Feb 2027, Quillon 4 Jul 2027 and Orrin 1 Aug 2027. Mirefield expires 31 May 2027.


## 9. Clean Up

While a run is **open**, the server can refuse to archive or delete the session: you get a 400 with `error_code: "workflow_run_open"`. To end runs early, send a `user.message` asking the agent to stop them; an interrupt alone ends no run. Ours has ended, so we can archive.

In [12]:
for file_id in uploaded.values():
    client.files.delete(file_id)

archived = client.beta.sessions.archive(session.id)
print(f"Session archived at {archived.archived_at}")

Session archived at 2026-10-11 15:03:51.897919+00:00


## Limits to Know

| Limit | Value | At the limit |
|---|---|---|
| Threads working at once in one run | 64 | The run waits until one finishes |
| Agents a workflow starts over the run's life | 1,000 | The run ends with `thread_limit_error` |
| Run lifetime | 24 hours, or what the agent sets | The run ends with `timeout_error`, even while paused |
| Runs open at once in a session | 10 | The start is refused (`max_workflow_runs_error`) |

Runs don't nest: only the primary thread's agent starts runs. A run's model requests count toward your Messages API rate limits.

## Summary

```
agents.create(multiagent={"type": "multiagent_20261001",
    "workflows": {"type": "enabled", "predefined_agents": [...]}})   →  the agent may write workflows
system prompt                    →  when to use a run, how to split the work, failure handling, time limit
sessions.create(budget=...)      →  one cap for the session and every run in it
user.message                     →  the agent decides to start a run (no API call does)
workflow_run.created / phase_* / status_ended  +  session.thread_created(workflow_run_id)
done = no open runs  AND  session.status_idle with end_turn
threads.list() → threads with workflow_run_id;  ask the agent for its workflow afterwards
```

The path through this cookbook led here:

| Module | What it contributed |
|---|---|
| 01–02 | Sessions, stream-first, multi-turn |
| 03 | Answer tool calls on arrival; read `stop_reason` |
| 04 | Files in, deliverables out |
| 05 | Budgets, interrupts, reconnecting |
| 06 | The `multiagent` block and session threads |
| **07** | A program that runs many agents, followed through its run, phases, and threads |

### Where to go next
- Scale up the data: put a few hundred contracts in `sample-data/` and watch the run fan out
- [Define outcomes](https://platform.claude.com/docs/en/managed-agents/define-outcomes): have a grader check the deliverable
- [Webhooks](https://platform.claude.com/docs/en/managed-agents/webhooks): get notified when a long session goes idle, instead of holding a stream open
- [Scheduled deployments](https://platform.claude.com/docs/en/managed-agents/scheduled-deployments): run a review like this one on a schedule